# lora_val_gen — generate the val slice at every checkpoint (per-step pipeline rebuild)
Generates the 98-row val slice for steps 1000/2000/3000/4000. The pipeline is rebuilt fresh for each step and one adapter is loaded — the exact single-adapter path proven in the test arm, no adapter swapping. Costs ~10 min extra per rank (four builds) but reuses only already-verified code. One notebook per rank; run three in parallel. Output: `gen/val/lora_r{RANK}__step{N}/`. Resumable per step. Sampler, negative, instruction handling identical to the test arm.

## 1. Repo

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's LoRA injection walks dispatch_torchao, which raises rather than returning False when torchao
# predates 0.16 (Colab ships 0.10). torchao is unused by this bf16 path; removing it lets injection
# proceed. Same fix the converter and training notebooks use. Guarded, so it is a no-op once gone.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

sys.path.insert(0, REPO)
os.chdir(REPO)

installed.
torchao removed


## 2. Drive, run config, adapter paths

In [ ]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "val"              # 98-row held-out slice for step selection
RANK    = 16                  # 4 | 8 | 16 — set per notebook
STEPS   = [1000, 2000, 3000, 4000]
RUN     = f"lora_r{RANK}"
EXP     = f"{DRIVE}/lora/experiments"

for step in STEPS:
    lora = f"{EXP}/{RUN}/checkpoint-{step}/transformer_lora/pytorch_lora_weights.safetensors"
    assert os.path.exists(lora), f"adapter missing, run lora_convert_steps first: {lora}"
print(f"rank {RANK}: all {len(STEPS)} adapters present")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rank 16: all 4 adapters present


## 3. Images, split files

In [ ]:
import subprocess, json
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"
def n_files(d): return sum(len(f) for _,_,f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train","val","test","screen","screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows, all reference images present")

restored 37063 / 37063
val: 98 rows, all reference images present


## 4. Engine

In [ ]:
from PIL import Image, ImageOps
import torch

# shipped default from inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

def generate_with(pipe, instruction, input_images=None, negative=NEG, seed=0, steps=50, text_g=5.0, image_g=2.0, size=512):
    gen = torch.Generator(device=accelerator.device).manual_seed(seed)
    res = pipe(prompt=instruction, input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=1024,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               negative_prompt=negative, num_images_per_prompt=1, generator=gen, output_type="pil",
               align_res=False)
    return res.images[0]

# no prompt lever in this arm: the adapter is the treatment, instruction and negative are the baseline
cfg = dict(instruction=lambda r: r["instruction"], negative=NEG)

## 5. Per-checkpoint build and pass (resumable)

In [ ]:
import torch, time, gc
from accelerate import Accelerator
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16

# per step: build the pipeline fresh and load that step's adapter (the exact proven single-adapter
# path from the test arm), generate, then free it. no adapter swapping.
def build_pipe(step):
    LORA_DIR = f"{EXP}/{RUN}/checkpoint-{step}/transformer_lora"
    pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
    pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
    pipe.transformer.load_lora_adapter(LORA_DIR, weight_name="pytorch_lora_weights.safetensors", prefix="transformer")
    pipe = pipe.to(accelerator.device)
    return pipe

def gen_step(step):
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    os.makedirs(GEN, exist_ok=True)
    have = {int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit()}
    todo = [i for i in range(len(rows)) if i not in have]
    print(f"\n=== step {step}: {len(have)} done, {len(todo)} to generate -> {GEN} ===")
    if not todo:
        return
    pipe = build_pipe(step)
    print(f"  pipeline built for step {step} on {accelerator.device}")
    try:
        t0 = time.time()
        for k, i in enumerate(todo):
            r = rows[i]
            refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
            img = generate_with(pipe, cfg["instruction"](r), input_images=refs or None, negative=cfg["negative"])
            img.save(f"{GEN}/{i:05d}.png")
            if (k+1) % 25 == 0:
                print(f"  {k+1}/{len(todo)}  ({(time.time()-t0)/(k+1):.1f}s/img)")
    finally:
        del pipe
        gc.collect(); torch.cuda.empty_cache()   # free VRAM before the next step's rebuild
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    print(f"  step {step}: {n}/{len(rows)} images")

for step in STEPS:
    gen_step(step)
print("\nall steps generated")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")



=== step 1000: 0 done, 98 to generate -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/lora_r16__step1000 ===


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  step 4000: 98/98 images

all steps generated


## 6. Manifests

In [ ]:
import json, torch, datetime
for step in STEPS:
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else 0
    if n == 0: continue
    man = {"dataset": DATASET, "run": RUN, "step": step, "n_images": n,
           "gpu": torch.cuda.get_device_name(0),
           "sampler": {"steps": 50, "text_guidance_scale": 5.0, "image_guidance_scale": 2.0,
                       "cfg_range": [0.0, 1.0], "max_sequence_length": 1024, "seed": 0, "size": 512, "align_res": False},
           "negative_prompt": NEG,
           "written": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(man, open(f"{GEN}/manifest.json", "w"), indent=2)
print("manifests written")

manifests written
